In [232]:
import pandas as pd
import numpy as np

In [233]:
df = pd.read_csv(
    "../data/raw/export-tranzactii-2022-sector-2.csv",
    sep=";"
)

df.shape

(4325, 10)

In [234]:
apartments = df[df["tip"] == "apartament"].copy()

apartments.shape

(3987, 10)

In [235]:
apartments = apartments.rename(columns={
    "tip": "property_type",
    "Tip strada": "street_type",
    "Denumire strada": "street_name",
    "Zona": "neighborhood",
    "S utila": "area_sqm",
    "S desfasurata": "built_area_sqm",
    "S teren": "land_area_sqm",
    "Pret tranzactie LEI": "price_lei",
    "an_constructie": "year_built",
    "Perioada tranzactie": "transaction_period"
})

apartments.columns

Index(['property_type', 'street_type', 'street_name', 'neighborhood',
       'area_sqm', 'built_area_sqm', 'land_area_sqm', 'price_lei',
       'year_built', 'transaction_period'],
      dtype='str')

In [236]:
apartments.head()

,property_type,street_type,street_name,neighborhood,area_sqm,built_area_sqm,land_area_sqm,price_lei,year_built,transaction_period
0,apartament,Alee,BISTRICIOARA,colentina,91.41,NaN,NaN,469908.0,1989.0,2022-T1
1,apartament,Alee,BISTRICIOARA,colentina,56.44,NaN,NaN,252241.0,1987.0,2022-T1
2,apartament,Alee,BRASOVENI,tei,52.40,NaN,NaN,373347.0,1985.0,2022-T1
3,apartament,Alee,BURIDAVA,colentina,43.56,NaN,NaN,316527.0,2016.0,2022-T1
4,apartament,Alee,CIRCULUI,tei,59.60,NaN,NaN,495000.0,1960.0,2022-T1


In [237]:
apartments["neighborhood"].value_counts(dropna=False)

neighborhood
centrala           841
pantelimon         717
tei                687
pipera sud         620
colentina          555
iancului           249
vatra luminoasa    117
floreasca           98
andronache          40
Colentina           27
ion creanga         19
-                   15
Tei                  2
Name: count, dtype: int64

In [238]:
apartments["neighborhood"] = (
    apartments["neighborhood"]
    .str.strip()
    .str.lower()
)

In [239]:
apartments["neighborhood"] = apartments["neighborhood"].replace("-", np.nan)

In [240]:
apartments["neighborhood"].value_counts(dropna=False)

neighborhood
centrala           841
pantelimon         717
tei                689
pipera sud         620
colentina          582
iancului           249
vatra luminoasa    117
floreasca           98
andronache          40
ion creanga         19
NaN                 15
Name: count, dtype: int64

In [241]:
apartments["year_built"].sort_values().tail(10)

1687    2022.0
2753    2022.0
1352    2022.0
1693    2022.0
1694    2022.0
1299    2022.0
4320    2022.0
4166    2087.0
2689       NaN
3748       NaN
Name: year_built, dtype: float64

In [242]:
apartments.loc[
    apartments["year_built"] > 2022,
    "year_built"
] = np.nan

In [243]:
apartments["year_built"].isna().sum()

np.int64(3)

In [244]:
apartments["year_built"].max()

np.float64(2022.0)

In [245]:
Q1 = apartments["area_sqm"].quantile(0.25)
Q3 = apartments["area_sqm"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

Q1, Q3, IQR, lower_bound, upper_bound

(np.float64(39.605),
 np.float64(65.305),
 np.float64(25.70000000000001),
 np.float64(1.0549999999999855),
 np.float64(103.85500000000002))

In [246]:
area_outliers = apartments[
    (apartments["area_sqm"] < lower_bound) |
    (apartments["area_sqm"] > upper_bound)
]

area_outliers.shape

(142, 10)

In [247]:
(apartments["area_sqm"] <= 0).sum()

np.int64(0)

In [248]:
apartments["area_sqm"].isna().sum()

np.int64(264)

In [249]:
apartments = apartments.dropna(subset=["area_sqm"])

apartments.shape

(3723, 10)

In [250]:
apartments["price_lei"].isna().sum()

np.int64(27)

In [251]:
(apartments["price_lei"] <= 0).sum()

np.int64(0)

In [252]:
apartments = apartments.dropna(subset=["price_lei"])

apartments.shape

(3696, 10)

In [253]:
apartments = apartments.drop(columns=[
    "property_type",
    "built_area_sqm",
    "land_area_sqm"
])

apartments.shape

(3696, 7)